In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colormaps
import warnings
warnings.filterwarnings('ignore')

# Caricamento dei dati
# Carica i dati degli incassi dei film - specificando il separatore di migliaia
incassi_df = pd.read_csv('IncassiTot.csv', thousands=',')
print("Dati incassi caricati:")
print(incassi_df.head())
print("\nInformazioni sui dati incassi:")
print(incassi_df.info())
print("\n" + "="*80 + "\n")

# Carica i dati dei prezzi medi dei biglietti
prezzi_df = pd.read_csv('PrezzoMedioTickets.csv')
print("Dati prezzi medi biglietti caricati:")
print(prezzi_df)
print("\n" + "="*80 + "\n")

# Pulizia dei dati incassi
# Rimuove eventuali spazi nei nomi delle colonne
incassi_df.columns = incassi_df.columns.str.strip()

# Verifica il tipo di dati nella colonna degli incassi
print("Tipo di dati nella colonna 'Incassi (in euro)':", incassi_df['Incassi (in euro)'].dtype)

# Se la colonna è già numerica, non è necessario convertire
if incassi_df['Incassi (in euro)'].dtype == 'object':
    # Se è stringa, rimuove le virgole e converte in float
    incassi_df['Incassi (in euro)'] = incassi_df['Incassi (in euro)'].str.replace(',', '').astype(float)
else:
    # Se è già numerica, la lascia così com'è
    print("La colonna degli incassi è già di tipo numerico")

# Estrae solo le colonne necessarie
incassi_df = incassi_df[['Nome del film', 'Incassi (in euro)']]

# Preparazione dei dati dei prezzi
# Estrae i prezzi per paese dalla riga dei dati
prezzi_per_paese = {}
for col in prezzi_df.columns:
    # Estrae il valore numerico dalla stringa (es: "10.8 eur" -> 10.8)
    # Gestisce diversi formati possibili
    valore_str = str(prezzi_df[col].iloc[0])
    # Rimuove "eur" o "€" e converte in float
    valore_str_clean = valore_str.replace('eur', '').replace('€', '').strip()
    prezzi_per_paese[col] = float(valore_str_clean)

print("Prezzi medi per paese:")
for paese, prezzo in prezzi_per_paese.items():
    print(f"{paese}: {prezzo} €")
print("\n" + "="*80 + "\n")

# Calcola i biglietti stimati per ogni film in ogni paese
# Assumiamo una distribuzione proporzionale basata sul PIL dei paesi
# Fonte: Stime approssimative basate su dati di mercato cinematografico
distribuzione_per_paese = {
    'America': 0.35,   # 35% del mercato globale
    'Cina': 0.25,      # 25% del mercato globale
    'Italia': 0.08,    # 8% del mercato globale
    'Spagna': 0.07,    # 7% del mercato globale
    'Regno Unito': 0.10, # 10% del mercato globale
    'Resto del mondo': 0.15 # 15% del mercato globale (non considerato nei prezzi)
}

# Normalizza la distribuzione per i paesi che abbiamo
paesi_disponibili = list(prezzi_per_paese.keys())
somma_distribuzione = sum([distribuzione_per_paese[p] for p in paesi_disponibili])
distribuzione_normalizzata = {p: distribuzione_per_paese[p]/somma_distribuzione for p in paesi_disponibili}

print("Distribuzione stimata del mercato (normalizzata per i paesi disponibili):")
for paese, percentuale in distribuzione_normalizzata.items():
    print(f"{paese}: {percentuale*100:.1f}%")
print("\n" + "="*80 + "\n")

# Crea un DataFrame per i biglietti stimati
biglietti_df = pd.DataFrame()

# Calcola i biglietti per ogni film e ogni paese
for idx, row in incassi_df.iterrows():
    film = row['Nome del film']
    incasso_totale = row['Incassi (in euro)']
    
    for paese in paesi_disponibili:
        # Calcola l'incasso per il paese
        incasso_paese = incasso_totale * distribuzione_normalizzata[paese]
        
        # Calcola il numero di biglietti (incasso / prezzo medio)
        prezzo_medio = prezzi_per_paese[paese]
        biglietti_paese = incasso_paese / prezzo_medio
        
        # Aggiunge al DataFrame
        biglietti_df.loc[film, paese] = biglietti_paese

# Converti in milioni di biglietti per una migliore leggibilità
biglietti_df_milioni = biglietti_df / 1_000_000

print("Biglietti stimati per film e paese (in milioni):")
print(biglietti_df_milioni.round(2))
print("\n" + "="*80 + "\n")

# Visualizzazione 1: Tabella dei biglietti stimati
plt.figure(figsize=(14, 8))
plt.subplot(1, 2, 1)

# Crea una tabella
cell_text = []
for film in biglietti_df_milioni.index:
    row = [f"{biglietti_df_milioni.loc[film, paese]:.2f}" for paese in biglietti_df_milioni.columns]
    cell_text.append(row)

table = plt.table(cellText=cell_text,
                  rowLabels=biglietti_df_milioni.index,
                  colLabels=[f"{paese}\n(milioni)" for paese in biglietti_df_milioni.columns],
                  cellLoc='center',
                  loc='center',
                  colWidths=[0.15 for _ in biglietti_df_milioni.columns])

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.2, 1.5)

plt.title('Biglietti stimati per film e paese (in milioni)', fontsize=14, pad=20)
plt.axis('off')

# Visualizzazione 2: Mappa di calore
plt.subplot(1, 2, 2)

# Prepara i dati per la heatmap
heatmap_data = biglietti_df_milioni.values

# Crea la heatmap
heatmap = plt.imshow(heatmap_data, cmap='YlOrRd', aspect='auto')

# Aggiungi le etichette
plt.xticks(range(len(biglietti_df_milioni.columns)), 
           [f"{paese}\n({prezzi_per_paese[paese]}€)" for paese in biglietti_df_milioni.columns], 
           rotation=45, ha='right')
plt.yticks(range(len(biglietti_df_milioni.index)), biglietti_df_milioni.index)

# Aggiungi i valori sulle celle
for i in range(len(biglietti_df_milioni.index)):
    for j in range(len(biglietti_df_milioni.columns)):
        plt.text(j, i, f'{heatmap_data[i, j]:.1f}M',
                ha='center', va='center',
                color='black' if heatmap_data[i, j] < np.max(heatmap_data)/2 else 'white',
                fontsize=9)

plt.colorbar(heatmap, label='Biglietti (milioni)')
plt.title('Mappa di calore: Biglietti stimati per film e paese', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

# Visualizzazione 3: Heatmap più dettagliata con Seaborn
plt.figure(figsize=(12, 8))
sns.heatmap(biglietti_df_milioni, 
            annot=True, 
            fmt='.1f',
            cmap='YlOrRd',
            linewidths=0.5,
            linecolor='gray',
            cbar_kws={'label': 'Biglietti venduti (milioni)'})

plt.title('Mappa di calore: Biglietti stimati per film e paese\n(Valori in milioni)', fontsize=16, pad=20)
plt.xlabel('Paese (con prezzo medio biglietto)', fontsize=12)
plt.ylabel('Film', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Statistiche riassuntive
print("="*80)
print("STATISTICHE RIASSUNTIVE")
print("="*80)

# Totale biglietti per film
print("\nTotale biglietti stimati per film (tutti i paesi, in milioni):")
totale_per_film = biglietti_df_milioni.sum(axis=1).sort_values(ascending=False)
for film, totale in totale_per_film.items():
    print(f"{film}: {totale:.2f} milioni")

# Totale biglietti per paese
print("\nTotale biglietti stimati per paese (tutti i film, in milioni):")
totale_per_paese = biglietti_df_milioni.sum(axis=0).sort_values(ascending=False)
for paese, totale in totale_per_paese.items():
    print(f"{paese}: {totale:.2f} milioni")

# Media biglietti per film
print(f"\nMedia biglietti per film: {biglietti_df_milioni.values.mean():.2f} milioni")
print(f"Film con più biglietti stimati: {totale_per_film.index[0]} ({totale_per_film.iloc[0]:.2f} milioni)")
print(f"Film con meno biglietti stimati: {totale_per_film.index[-1]} ({totale_per_film.iloc[-1]:.2f} milioni)")

# Esporta i risultati in un file CSV
biglietti_df_milioni.to_csv('biglietti_stimati_per_film_paese.csv')
print("\n" + "="*80)
print(f"Dati esportati in 'biglietti_stimati_per_film_paese.csv'")